# Suite XTY — The Cpp Types dialect

`Expressions.Cpp.Types` models C++'s type-ids (ISO/IEC 14882, [dcl.name]), a grammar of their own, as mbse-programs has
them: a type-id, its specifiers and its abstract declarator, C++'s own helpers, which a lambda's declarator holds too,
and an enumeration's body, its enumerators and the directives and comments among them, as data. C++'s terms hold its
terms wherever a type stands (a cast's, a named cast's, `new`'s, a conversion function's name's, a trailing return
type, an enumeration's fixed type, a type parameter's default) and where an expression or a type may (`sizeof(int)`),
C++ reading them. Its kinds are validated, bind nothing, cover C++'s three editions, print as mbse-programs prints
them, and are refused by evaluation by name; a cast reads its type from it.

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as CFamily from "@mbse/expressions/C/Expressions";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor";
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppFamily from "@mbse/expressions/Cpp/Expressions";
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import { Expressions as XR } from "@mbse/expressions/Cpp/RegExp";
import * as AMD64 from "@mbse/expressions/SystemV/AMD64";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators } from "@mbse/schemas/Framework";
import { assert, equal, raises, same } from "./support.js";

const { LookupError, NotImplementedError, ValueError } = Errors;
const [C, PP] = [CFamily.Expressions, CPreprocessor.Expressions];
const [CPP, XV, XText, XT, XTText] = [CppFamily.Expressions, CppFamily.Evaluators, CppFamily.Text, CppTypes.Expressions, CppTypes.Text];

const store: any = new Builders.Store(); // the C family's dialects, Cpp Types' install before C++'s: the installs run in any order
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, CppTypes.install, CFamily.install, CppFamily.install,
  CTypes.install, CPreprocessor.install, AMD64.install]) {
  store.transaction(install);
}
for (const define of [CFamily.define, CppFamily.define, AMD64.define, CppTypes.define]) store.transaction(define); // C++'s types and the ABI's, its editions
const AMD64_ = store.Instances.Expressions.SystemV.AMD64; // the ABI a holder names, whose types C++ leaves to the implementation

/** The term value `spec` writes, the root of `dialect`'s holder, naming `abi`, written in a transaction. */
function term(spec: unknown, dialect: F.Dialect = XT.DIALECT, abi: any = AMD64_): any {
  return F.term(store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec)).abi([abi]))));
}

const validate = (spec: unknown, bound: string[] = [], dialect: F.Dialect = XT.DIALECT): string[] =>
  dialect.validate(term(spec, dialect), bound);
const text = (spec: unknown, dialect: F.Dialect = XT.DIALECT): string => XTText.ToText(term(spec, dialect));
const union = (name: string): string[] => store.registered(name).branches.map((b: any) => b.name); // a union's branches
const schemas = (name: string): Map<string, any> => new Map(store.registered(name).branches.map((b: any) => [b.name, b.schema]));
const spelled = (word: string) => (i: any) => i.spelling(word); // an identifier, of its spelling
const named = (word: string) => (n: any) => n.identifier(spelled(word)); // a name that is an identifier
const keyword = (word: string) => (s: any) => s.primitivetypespecifier((p: any) => p.keyword(word));
/** A type-id, of its specifiers, and its declarator where given. */
const typeName = (specifiers: unknown[], declarator: unknown = null) => (t: any) => t.typeid((i: any) =>
  (declarator === null ? i.specifiers(specifiers) : i.specifiers(specifiers).declarator(declarator)));
const int_ = typeName([keyword("int")]);
const x = CPP.identifier("x");
/** `word = value`. */
const enumerator = (word: string, value: unknown = null) => (u: any) => u.enumerator(
  (r: any) => (value === null ? r.name(spelled(word)) : r.name(spelled(word)).value(value)));
/** `enum class E { items }`. */
const enumeration = (items: unknown[], key = "enum class", base: unknown = null) => typeName([(s: any) => s.enumspecifier(
  (e: any) => (base === null ? e.key(key).name(named("E")) : e.key(key).name(named("E")).base(base)).body(true).enumerators(items))]);


<a id="XTY-01"></a>
## XTY-01 · Its union and holder: its kinds, a type-id and an enumerator, invalid syntax, and the preprocessor's directives and comment; a holder refining the framework's, naming its ABI, round-tripped through JSON

In [ ]:
{
  assert(same(XT.TAGS, ["typeid", "enumerator"]) && same([...XT.DIALECT.kinds().keys()], [...XT.TAGS, "error", "missing"]) && XT.HELPERS.length === 0);
  assert(same(union(XT.DIALECT.union), [...XT.TAGS, "error", "missing", ...PP.DIRECTIVES, "comment"]));
  assert(XT.DIALECT.name() === "Cpp Types" && XT.DIALECT.tag === "cpp_types" && union("Expressions.Framework.Term").includes("cpp_types"));
  const [kinds, held] = [schemas(XT.DIALECT.union), schemas(CPP.DIALECT.union)];
  assert(held.get("typeid") === kinds.get("typeid") && kinds.get("typeid") === store.registered("Expressions.Cpp.Types.TypeId")); // one schema, in each union
  assert(kinds.get("typeid") !== schemas("Expressions.C.Types.Term").get("typeid")); // C++'s own, holding C++'s helpers
  assert(same(XT.DIALECT.reads(), [CPP.DIALECT, PP.DIALECT, XR.DIALECT]) && CPP.DIALECT.kindNamed("typeid") === XT.DIALECT.kinds().get("typeid"));
  const Holder: any = store.registered("Expressions.Cpp.Types.Expression");
  assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(XT.DIALECT.union) && same([...Holder.adjacencies.keys()], ["abi"]));
  const holder = store.transaction(() => store.Expressions.Cpp.Types.Expression((e: any) => e.root(XT.DIALECT.spec(enumeration([enumerator("A")]))).abi([AMD64_])));
  assert(Validators.Validate(store)(Holder, holder).length === 0 && XT.DIALECT.validate(holder).length === 0);
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)) as any).root, holder.root));
  raises(Error, () => store.transaction(() => // its root holds its own terms
    store.Expressions.Cpp.Types.Expression((e: any) => e.root(XT.DIALECT.spec(CPP.binary("+", x, 1n))))),
  "'binaryexpression' is not a member of a union value");
}


<a id="XTY-02"></a>
## XTY-02 · Its kinds and positions: a type-id's parts C++'s helpers, read through reads; an enumeration's body its terms; every position of a type in C++ holding its Term

In [ ]:
{
  const props = (path: string, name: string): string[] => [...store.registered(`Expressions.${path}.${name}`).properties.keys()];
  assert(same(props("Cpp.Types", "TypeId"), ["specifiers", "declarator"]) && same(props("Cpp.Types", "Enumerator"), ["name", "attributes", "value"]));
  assert(same(props("Cpp.Expressions", "EnumSpecifier"), ["key", "attributes", "name", "base", "body", "enumerators", "trailing_comma"]));
  const prop = (path: string, name: string, member: string): any => store.registered(`Expressions.${path}.${name}`).properties.get(member).schema;
  assert(prop("Cpp.Types", "TypeId", "specifiers").element === store.registered("Expressions.Cpp.Expressions.Specifier"));
  assert(prop("Cpp.Types", "TypeId", "declarator") === store.registered("Expressions.Cpp.Expressions.Declarator"));
  assert(prop("Cpp.Types", "Enumerator", "value") === store.registered(CPP.DIALECT.union)); // C++'s terms
  assert(prop("Cpp.Types", "Enumerator", "name") === store.registered("Expressions.Cpp.Expressions.Identifier"));
  assert(prop("Cpp.Expressions", "EnumSpecifier", "enumerators").element === store.registered(XT.DIALECT.union));
  const Types = store.registered(XT.DIALECT.union);
  for (const [name, member] of [["CastExpression", "type"], ["NamedCastExpression", "type"], ["NewExpression", "type"],
    ["ConversionName", "type"], ["LambdaDeclarator", "trailing_return"], ["FunctionDeclarator", "trailing_return"],
    ["EnumSpecifier", "base"], ["TypeParameter", "default"]] as [string, string][]) {
    assert(prop("Cpp.Expressions", name, member) === Types, name);
  }
  raises(LookupError, () => store.registered("Expressions.Cpp.Expressions.Type"), "no schema registered as 'Expressions.Cpp.Expressions.Type'"); // moved, no alias
  raises(LookupError, () => store.registered("Expressions.Cpp.Expressions.TypeId"), "no schema registered as 'Expressions.Cpp.Expressions.TypeId'");
}


<a id="XTY-03"></a>
## XTY-03 · Its checks: an enumerator needs a name, an identifier's spelling, an enumeration's enumerators within its body, and an enumeration's items of C++ alone

In [ ]:
{
  assert(same(validate((u: any) => u.enumerator((r: any) => r)), ["an enumerator needs a name"]));
  assert(same(validate(enumerator("a b")), ["'a b' is not an identifier"]));
  assert(same(validate(typeName([(s: any) => s.enumspecifier((e: any) => e.key("enum").enumerators([enumerator("A")]))])), [
    "an EnumSpecifier with enumerators has a body"]));
  assert(same(validate(enumerator("A", CPP.literal(1n, "wb"))), ["value: a literal's suffix cannot be 'wb'"])); // C++'s suffixes
  const cItem = CTypes.Expressions.DIALECT.spec((u: any) => u.enumerator((r: any) => r.name(spelled("B"))));
  const directive = XT.DIALECT.spec((u: any) => u.ifdefdirective((d: any) => d.name(spelled("B_")).items([cItem])));
  assert(same(validate(enumeration([directive])), ["specifiers 0 argument 0: argument 0: a C Types enumerator is not a Cpp Types term"]));
  const cCondition = XT.DIALECT.spec((u: any) => u.ifdirective((d: any) => d.condition(C.binary(">", C.identifier("N"), 1n))));
  assert(same(validate(enumeration([cCondition])), ["specifiers 0 argument 0: condition: a C binaryexpression is not a Cpp Types term"]));
}


<a id="XTY-04"></a>
## XTY-04 · What it binds: nothing; a lambda's parameters within its trailing return type, a C++ binder's own

In [ ]:
{
  assert(!store.registered(XT.DIALECT.union).branches.some((b: any) => F.isBinder(b.schema)));
  assert(same([...SY.free(term(enumeration([enumerator("A", x)])), XT.DIALECT)], ["x"]));
  const decltype = (word: string) => typeName([(s: any) => s.decltypespecifier((d: any) => d.expression(CPP.identifier(word)))]);
  const lambda_ = (u: any) => u.lambdaexpression((r: any) => r.declarator((d: any) => d.parameters([ // [](int a) -> decltype(a) {}
    (p: any) => p.parameterdeclaration((q: any) => q.specifiers([keyword("int")]).declarator((e: any) => e.iddeclarator(
      (i: any) => i.name(named("a")))))]).trailing_return(decltype("a"))));
  assert(validate(lambda_, [], CPP.DIALECT).length === 0 && XText.ToText(term(lambda_, CPP.DIALECT)) === "[](int a) -> decltype(a) {}");
  assert(SY.free(term(lambda_, CPP.DIALECT), CPP.DIALECT).size === 0);
  assert(same(validate((u: any) => u.lambdaexpression((r: any) => r.declarator((d: any) => d.trailing_return(decltype("a")))), [], CPP.DIALECT), [
    "declarator trailing_return: specifiers 0 expression: idexpression 'a' is not bound"]));
}


<a id="XTY-05"></a>
## XTY-05 · Within C++'s terms: a type wherever one stands, a Cpp Types term C++ reads, validates, infers, evaluates and prints; written by C++'s writers

In [ ]:
{
  const uint8 = typeName([(s: any) => s.namedtypespecifier((n: any) => n.name(named("uint8_t")))]);
  const cast = term(CPP.cast(uint8, CPP.binary("+", x, 1n)), CPP.DIALECT);
  assert(same(F.form(cast).arguments.map((a: any) => F.dialectOf(a)), [XT.DIALECT.union, CPP.DIALECT.union]));
  assert(CPP.DIALECT.validate(cast, ["x"]).length === 0 && XText.ToText(cast) === "(uint8_t)(x + 1)" && equal(XV.OfAny(cast, { x: 255n }), 0n));
  assert(CPP.DIALECT.infer(cast, { x: D.Anything }).name() === "uint8_t" && F.same(term(CPP.DIALECT.make(F.form(cast)), CPP.DIALECT), cast));
  const cases: [unknown, string][] = [
    [(u: any) => u.namedcastexpression((n: any) => n.operator("static_cast").type(int_).operand(x)), "static_cast<int>(x)"],
    [(u: any) => u.newexpression((n: any) => n.type(int_)), "new int"],
    [(u: any) => u.sizeofexpression((s: any) => s.operand(int_)), "sizeof(int)"],
    [(u: any) => u.idexpression((r: any) => r.name((n: any) => n.conversionname((c: any) => c.type(int_)))), "operator int"],
    [(u: any) => u.lambdaexpression((r: any) => r.template_parameters([(t: any) => t.typeparameter((p: any) => p.key("typename").name(
      spelled("T")).default(int_))])), "[]<typename T = int> {}"],
    [(u: any) => u.sizeofexpression((s: any) => s.operand(enumeration([enumerator("A")], "enum", int_))), "sizeof(enum E : int {\n    A\n})"],
  ];
  for (const [spec, source] of cases) {
    assert(CPP.DIALECT.validate(term(spec, CPP.DIALECT), ["x"]).length === 0 && XText.ToText(term(spec, CPP.DIALECT)) === source, source);
  }
  assert(same(CPP.DIALECT.validate(term((u: any) => u.newexpression((n: any) => n), CPP.DIALECT)), ["a newexpression needs a type"]));
}


<a id="XTY-06"></a>
## XTY-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
{
  assert(same(validate((u: any) => u.error((r: any) => r.text("int int"))), ["invalid syntax 'int int'"]) && text((u: any) => u.error((r: any) => r.text("a b"))) === "a b");
  assert(same(validate((u: any) => u.missing((r: any) => r.kind("type"))), ["missing type"]));
  const typed = term((u: any) => u.namedcastexpression((n: any) => n.operator("static_cast").type((t: any) => t.missing((m: any) => m.kind("type"))).operand(x)), CPP.DIALECT);
  assert(same(CPP.DIALECT.validate(typed, ["x"]), ["type: missing type"]) && XText.ToText(typed) === "static_cast<>(x)");
  raises(ValueError, () => text((u: any) => u.error((r: any) => r)), "Cpp Types error has no text to print");
}


<a id="XTY-07"></a>
## XTY-07 · Its editions: C++'s three, each by its designation and common name, each with both its kinds

In [ ]:
{
  const types = store.Instances.Expressions.Cpp.Types;
  assert(equal([...FS.standards(types).values()].map((s) => FS.tags(s)), [
    ["ISO/IEC 14882:2017", "C++17"], ["ISO/IEC 14882:2020", "C++20"], ["ISO/IEC 14882:2024", "C++23"]]));
  assert(equal(FS.properties(FS.find(types, "C++23")!).get("revision"), ["year", 2024n]) && XT.SINCE.size === 0);
  assert([...FS.standards(types).values()].every((e) => same(FS.kinds(e).map((s: any) => s.name),
    ["Expressions.Cpp.Types.TypeId", "Expressions.Cpp.Types.Enumerator"])));
  assert(store.transaction(CppTypes.define)[0].get("Cpp20") === FS.find(types, "C++20")); // run again, it writes nothing
}


<a id="XTY-08"></a>
## XTY-08 · Printing: each type-id as mbse-programs' printer writes it, an enumeration's body on lines of its own

In [ ]:
{
  const reference = (d: any) => d.referencedeclarator((r: any) => r);
  const cases: [unknown, string][] = [
    [int_, "int"], [typeName([(s: any) => s.cvqualifier((q: any) => q.keyword("const")), (s: any) => s.namedtypespecifier(
      (n: any) => n.name((m: any) => m.qualifiedname((q: any) => q.qualifiers([named("std")]).name(named("string")))))], reference),
      "const std::string&"],
    [enumeration([enumerator("A", CPP.literal(1n)), enumerator("B")]), "enum class E {\n    A = 1,\n    B\n}"],
    [enumeration([enumerator("A"), (u: any) => u.comment((c: any) => c.text(" B")), XT.DIALECT.spec((u: any) => u.otherdirective(
      (d: any) => d.directive("pragma").text("region"))), enumerator("C")]),
      "enum class E {\n    A,\n    // B\n#pragma region\n    C\n}"],
    [enumerator("A", CPP.literal(1n)), "A = 1"],
  ];
  for (const [spec, source] of cases) assert(text(spec) === source, `${source} != ${text(spec)}`);
}


<a id="XTY-09"></a>
## XTY-09 · Evaluation: none (Q11); each of its kinds is refused by name by its family's evaluator; a cast reads its type rather than evaluate it

In [ ]:
{
  for (const [spec, tag] of [[int_, "typeid"], [enumerator("A"), "enumerator"]] as [unknown, string][]) {
    raises(NotImplementedError, () => XV.OfAny(term(spec), {}), `Cpp Types ${tag} is not evaluated`);
  }
  assert(equal(XV.OfAny(term(CPP.cast(typeName([keyword("signed"), keyword("char")]), 200n), CPP.DIALECT)), -56n));
  raises(NotImplementedError, () => XV.OfAny(term((u: any) => u.namedcastexpression((n: any) => n.operator("static_cast").type(int_).operand(CPP.literal(1n))), CPP.DIALECT)),
    "Cpp namedcastexpression is not evaluated");
}
